# Demo 2: Change frequency with measurement meaning

**Learning question:** When should a temporal panel conform to a grid, and when should observations be summarized into bins?

The input grain is **one recorded station observation per row**. The hourly output grain is one station-hour grid label per row; the second output grain is one station--two-hour interval per row. This required demo is Colab-first and runs equivalently in local Jupyter or VS Code. Colab storage is ephemeral, and changes opened from GitHub are not automatically saved back to the repository.

Use only the supplied synthetic, non-identifying fixture. Do not add credentials, private records, manual uploads, or Drive mounts. Restart the kernel and run every cell in order; stored output is not execution evidence. Assignment Colab support remains conditional on the repository-save and Classroom50 pilot.


In [ ]:
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

PYTHON_CANDIDATE = "3.12.13"
NUMPY_CANDIDATE = "2.0.2"
PANDAS_CANDIDATE = "3.0.3"
COURSE_PACKAGES = {"numpy": NUMPY_CANDIDATE, "pandas": PANDAS_CANDIDATE}


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


mismatched = [
    f"{package_name}=={candidate}"
    for package_name, candidate in COURSE_PACKAGES.items()
    if installed_version(package_name) != candidate
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

import numpy as np
import pandas as pd

assert platform.python_version() == PYTHON_CANDIDATE
assert np.__version__ == NUMPY_CANDIDATE
assert pd.__version__ == PANDAS_CANDIDATE
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## Define frequency change and missingness provenance

This notebook repeats Demo 1's exact parsing, localization, UTC conversion, and station/time sort so it is independently restartable.

**Upsampling** requests a finer grid; **downsampling** creates coarser bins. `asfreq()` conforms observations to exact grid labels without combining them. `resample()` groups timestamps into bins and requires a summary when multiple readings can contribute.

**Source-value missingness** means a source row exists but its measurement is missing. A **grid-created row** is a requested label for which no source row exists. Both can display a missing temperature, but their provenance differs. A grid change alone does not justify forward fill, backward fill, interpolation, or zero.

**Measurement meaning** states what a value represents and how it may be combined. Temperature is a state observed at an instant; its bin mean answers a bounded recorded-temperature question. The source-row marker is an additive reading counter. A **left-closed, left-labeled bin** includes its left boundary and uses that boundary as its label.


In [ ]:
from hashlib import sha256
from pathlib import Path

EXPECTED_FIXTURE_SHA256 = "57dcdb82372805cf1dda83a7c227b463fe997cf1437275d64d01b9719ff26b54"
FIXTURE_BYTES = (
    b"station,observed_at,temperature_c\n"
    b"south,2026-01-15 13:00,23.0\n"
    b"north,2026-01-15 08:00,10.0\n"
    b"south,2026-01-15 08:00,20.0\n"
    b"north,2026-01-15 14:00,14.0\n"
    b"south,2026-01-15 10:00,21.0\n"
    b"north,2026-01-15 11:00,\n"
    b"south,2026-01-15 14:00,24.0\n"
    b"north,2026-01-15 09:00,11.0\n"
    b"south,2026-01-15 11:00,22.0\n"
    b"north,2026-01-15 12:00,13.0\n"
)


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "09" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATH = DATA_DIRECTORY / "station_observations.csv"
if not FIXTURE_PATH.exists():
    FIXTURE_PATH.write_bytes(FIXTURE_BYTES)

actual_fixture_sha256 = sha256(FIXTURE_PATH.read_bytes()).hexdigest()
assert actual_fixture_sha256 == EXPECTED_FIXTURE_SHA256, (
    "station_observations.csv does not match the supplied fixture checksum. "
    "Restore the committed file; corrupt data are never replaced silently."
)

raw = pd.read_csv(
    FIXTURE_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "temperature_c": "float64",
    },
)
raw["source_row"] = np.int64(1)

assert raw.shape == (10, 4)
assert raw["station"].dtype == pd.StringDtype()
assert raw["observed_at"].dtype == pd.StringDtype()
assert raw["temperature_c"].dtype == np.dtype("float64")
assert raw["source_row"].dtype == np.dtype("int64")
assert raw["temperature_c"].isna().sum() == 1

naive_times = pd.to_datetime(
    raw["observed_at"],
    format="%Y-%m-%d %H:%M",
)
assert naive_times.dt.tz is None
aware_times = naive_times.dt.tz_localize("America/Los_Angeles")
raw["observed_at"] = aware_times.dt.tz_convert("UTC")

prepared = raw.sort_values(
    ["station", "observed_at"],
    kind="stable",
).reset_index(drop=True)

assert prepared["station"].dtype == pd.StringDtype()
assert str(prepared["observed_at"].dtype) == "datetime64[us, UTC]"
assert prepared["temperature_c"].dtype == np.dtype("float64")
assert prepared["source_row"].dtype == np.dtype("int64")
assert prepared.shape == (10, 4)
assert not prepared.duplicated(["station", "observed_at"]).any()
assert all(
    group["observed_at"].is_monotonic_increasing
    for _, group in prepared.groupby(
        "station", observed=True, sort=True, dropna=True
    )
)


OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OWNED_OUTPUT_NAMES = ['hourly_grid.csv', 'two_hour_summary.csv']
for output_name in OWNED_OUTPUT_NAMES:
    output_path = OUTPUT_DIRECTORY / output_name
    if output_path.exists():
        output_path.unlink()


def write_verified_csv(frame, path, *, expected_size, expected_sha256):
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep="",
    )
    first_bytes = path.read_bytes()
    assert len(first_bytes) == expected_size
    assert sha256(first_bytes).hexdigest() == expected_sha256
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep="",
    )
    assert path.read_bytes() == first_bytes
    return first_bytes


def restore_utc_timestamp(frame, column="observed_at"):
    frame[column] = pd.to_datetime(
        frame[column],
        format="%Y-%m-%d %H:%M:%S%z",
        utc=True,
    )
    return frame


print("Demo directory:", DEMO_DIRECTORY)
print("Fixture SHA-256:", actual_fixture_sha256)


In [ ]:
indexed_panel = prepared.set_index("observed_at")
hourly_grid = (
    indexed_panel.groupby(
        "station",
        observed=True,
        sort=True,
        dropna=True,
    )[["temperature_c", "source_row"]]
    .resample("h")
    .asfreq()
    .reset_index()
)
hourly_grid["grid_created_row"] = hourly_grid["source_row"].isna()
hourly_grid["source_value_missing"] = (
    hourly_grid["source_row"].eq(1)
    & hourly_grid["temperature_c"].isna()
)

assert hourly_grid.shape == (14, 6)
assert hourly_grid["station"].dtype == pd.StringDtype()
assert str(hourly_grid["observed_at"].dtype) == "datetime64[us, UTC]"
assert hourly_grid["temperature_c"].dtype == np.dtype("float64")
assert hourly_grid["source_row"].dtype == np.dtype("float64")
assert hourly_grid["grid_created_row"].dtype == np.dtype("bool")
assert hourly_grid["source_value_missing"].dtype == np.dtype("bool")
assert hourly_grid["grid_created_row"].sum() == 4
assert hourly_grid["source_value_missing"].sum() == 1
assert not (
    hourly_grid["grid_created_row"]
    & hourly_grid["source_value_missing"]
).any()
assert hourly_grid["source_row"].notna().sum() == len(prepared)
assert set(hourly_grid["station"]) == {"north", "south"}

created_pairs = list(
    hourly_grid.loc[
        hourly_grid["grid_created_row"],
        ["station", "observed_at"],
    ].itertuples(index=False, name=None)
)
assert created_pairs == [
    ("north", pd.Timestamp("2026-01-15 18:00", tz="UTC")),
    ("north", pd.Timestamp("2026-01-15 21:00", tz="UTC")),
    ("south", pd.Timestamp("2026-01-15 17:00", tz="UTC")),
    ("south", pd.Timestamp("2026-01-15 20:00", tz="UTC")),
]
missing_source = hourly_grid.loc[hourly_grid["source_value_missing"]]
assert missing_source[["station"]].to_numpy().tolist() == [["north"]]
assert missing_source["observed_at"].iloc[0] == pd.Timestamp(
    "2026-01-15 19:00", tz="UTC"
)

print(hourly_grid)


In [ ]:
HOURLY_OUTPUT_PATH = OUTPUT_DIRECTORY / "hourly_grid.csv"
hourly_bytes = write_verified_csv(
    hourly_grid,
    HOURLY_OUTPUT_PATH,
    expected_size=788,
    expected_sha256="7054dfb410b36f35ef53ff4e02cc77fb633ff413e1dcd9f66d1807674053b40e",
)
hourly_readback = pd.read_csv(
    HOURLY_OUTPUT_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "temperature_c": "float64",
        "source_row": "float64",
        "grid_created_row": "bool",
        "source_value_missing": "bool",
    },
)
restore_utc_timestamp(hourly_readback)
assert hourly_readback["station"].dtype == pd.StringDtype()
assert str(hourly_readback["observed_at"].dtype) == "datetime64[us, UTC]"
assert hourly_readback["temperature_c"].dtype == np.dtype("float64")
assert hourly_readback["source_row"].dtype == np.dtype("float64")
assert hourly_readback["grid_created_row"].dtype == np.dtype("bool")
assert hourly_readback["source_value_missing"].dtype == np.dtype("bool")
pd.testing.assert_frame_equal(hourly_readback, hourly_grid)

print("Wrote:", HOURLY_OUTPUT_PATH)
print("Hourly SHA-256:", sha256(hourly_bytes).hexdigest())


In [ ]:
two_hour_summary = (
    indexed_panel.groupby(
        "station",
        observed=True,
        sort=True,
        dropna=True,
    )
    .resample("2h", closed="left", label="left")
    .agg(
        mean_temperature_c=("temperature_c", "mean"),
        reading_count=("source_row", "sum"),
    )
    .reset_index()
)

assert two_hour_summary.shape == (8, 4)
assert two_hour_summary["station"].dtype == pd.StringDtype()
assert str(two_hour_summary["observed_at"].dtype) == "datetime64[us, UTC]"
assert two_hour_summary["mean_temperature_c"].dtype == np.dtype("float64")
assert two_hour_summary["reading_count"].dtype == np.dtype("int64")
assert two_hour_summary["station"].tolist() == ["north"] * 4 + ["south"] * 4
assert two_hour_summary["reading_count"].tolist() == [2, 1, 1, 1, 1, 2, 1, 1]
assert int(two_hour_summary["reading_count"].sum()) == len(prepared)
expected_means = [10.5, np.nan, 13.0, 14.0, 20.0, 21.5, 23.0, 24.0]
np.testing.assert_allclose(
    two_hour_summary["mean_temperature_c"].to_numpy(),
    np.asarray(expected_means),
    equal_nan=True,
)
expected_bins = [
    pd.Timestamp(f"2026-01-15 {hour:02d}:00", tz="UTC")
    for hour in (16, 18, 20, 22)
] * 2
assert two_hour_summary["observed_at"].tolist() == expected_bins

print(two_hour_summary)


In [ ]:
SUMMARY_OUTPUT_PATH = OUTPUT_DIRECTORY / "two_hour_summary.csv"
summary_bytes = write_verified_csv(
    two_hour_summary,
    SUMMARY_OUTPUT_PATH,
    expected_size=361,
    expected_sha256="0558659b66336e71c3c67769097aadf4e2616a2d4f913425bb498463528a9d6f",
)
summary_readback = pd.read_csv(
    SUMMARY_OUTPUT_PATH,
    dtype={
        "station": "string",
        "observed_at": "string",
        "mean_temperature_c": "float64",
        "reading_count": "int64",
    },
)
restore_utc_timestamp(summary_readback)
assert summary_readback["station"].dtype == pd.StringDtype()
assert str(summary_readback["observed_at"].dtype) == "datetime64[us, UTC]"
assert summary_readback["mean_temperature_c"].dtype == np.dtype("float64")
assert summary_readback["reading_count"].dtype == np.dtype("int64")
pd.testing.assert_frame_equal(summary_readback, two_hour_summary)

demo2_verified = True
print("Wrote:", SUMMARY_OUTPUT_PATH)
print("Summary SHA-256:", sha256(summary_bytes).hexdigest())


## Interpret the frequency changes

The hourly grid adds labels but does not invent measurements. North 19:00 is a supplied row with a missing temperature; North 18:00 is a new grid label with no supplied row. The two-hour summary answers a different question by combining recorded values inside explicit station-specific bins. North's 18:00-bin mean is missing because its one recorded temperature is missing, not because the station interval is empty.


In [ ]:
assert demo2_verified is True
assert sha256(FIXTURE_PATH.read_bytes()).hexdigest() == EXPECTED_FIXTURE_SHA256
assert sha256(HOURLY_OUTPUT_PATH.read_bytes()).hexdigest() == "7054dfb410b36f35ef53ff4e02cc77fb633ff413e1dcd9f66d1807674053b40e"
assert sha256(SUMMARY_OUTPUT_PATH.read_bytes()).hexdigest() == "0558659b66336e71c3c67769097aadf4e2616a2d4f913425bb498463528a9d6f"
assert hourly_grid["grid_created_row"].sum() == 4
assert hourly_grid["source_value_missing"].sum() == 1
assert int(two_hour_summary["reading_count"].sum()) == 10
print("Lecture 09 Demo 2 fresh-execution verification passed.")
